# Regression robustness ladder: full coverage, spec curve, Oster bounds

Standalone companion to `regression.ipynb`. Does not modify it or any other tracked
file - it only reads `scripts/model_helpers.py` (unchanged) and the same processed
dataset, and writes to `outputs/tables/` and `outputs/standardized_tables/` using the
same "outcome | Model label" naming convention `regression.ipynb` already established
for Models C1-C5, S (saturated), and O (over-controlled).

`regression.ipynb` currently has the cumulative ladder for two outcomes (`y_pv`,
`y_storage`). This notebook:

1. Fits the same seven-model ladder for **all ten outcomes** (the two that overlap
   come out numerically identical, since both notebooks use the same formulas and
   dataset - confirmed byte-identical before this notebook was added).
2. Runs a **specification curve**: every subset of the six confounder blocks
   (2^6 = 64, minus the 16 that stack county FE with utility FE - both are
   near-exhaustive, near-collinear geographic/institutional partitions of the same
   ZIPs), raw scale, for the four focal terms. -> `outputs/tables/spec_curve.csv`.
3. Computes **Oster (2019) delta** bounds, Model C1 (restricted) vs. Model S (full),
   for the same four focal terms across all ten outcomes.
   -> `outputs/tables/oster_delta.csv`.

See `REGRESSION_ROBUSTNESS_PLAN.md` and `REGRESSION_ROBUSTNESS_RESULTS.html` for the
full design rationale and results discussion.

In [ ]:
import itertools
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import patsy
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

SCRIPTS_DIR = (Path.cwd() / "scripts" if (Path.cwd() / "scripts").exists() else Path.cwd().parent / "scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

# All from main's unchanged scripts/model_helpers.py - this notebook adds no functions
# there, so it introduces no diff against that file.
from model_helpers import common_sample_index, fit_stats, pv_kw_per_1000, run_ols, vif_from_formula

from oster_delta_helpers import oster_delta

In [ ]:
OUTCOMES_TO_RUN = [
    "y_pv",
    "y_storage",
    "y_chargers",
    "y_wind_mw",
    "any_turbines",
    "y_level1_chargers",
    "y_level2_chargers",
    "y_dc_fast_chargers",
    "energy_burden_pct",
    "log_energy_gap_per_capita",
]
RUN_BOTH_OUTPUT_MODES = True
RUN_SPEC_CURVE = True

OUTPUT_TABLE_DIRS = {
    False: Path("../outputs/tables"),
    True: Path("../outputs/standardized_tables"),
}
OSTER_DELTA_PATH = Path("../outputs/tables/oster_delta.csv")
SPEC_CURVE_PATH = Path("../outputs/tables/spec_curve.csv")

In [ ]:
def prep_outcomes_per_capita(df, pop_col="total_population", min_pop=1000):
    """Create per-capita + log1p outcomes; filter tiny-pop ZIPs.

    Identical to the corresponding function in regression.ipynb - duplicated here so
    this notebook has no import dependency on that one, and stays self-contained if
    regression.ipynb changes shape in the future.
    """
    df = df.copy()
    df[pop_col] = pd.to_numeric(df[pop_col], errors="coerce")
    der_zero_cols = [
        "PV_system_size_DC", "total_chargers", "level1_chargers", "level2_chargers",
        "dc_fast_chargers", "zev_count", "plant_capacity_mw", "storage_capacity_mw",
        "wind_capacity_mw", "wind_turbine_count",
    ]
    for col in der_zero_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df = df[df[pop_col].notna() & (df[pop_col] >= min_pop)].copy()

    pop = df[pop_col].replace(0, np.nan)
    df["level2_chargers_per_1k"] = df["level2_chargers"] * 1000 / pop
    df["y_level2_chargers"] = np.log1p(df["level2_chargers_per_1k"])
    df["level1_chargers_per_1k"] = df["level1_chargers"] * 1000 / pop
    df["y_level1_chargers"] = np.log1p(df["level1_chargers_per_1k"])
    df["dc_fast_chargers_per_1k"] = df["dc_fast_chargers"] * 1000 / pop
    df["y_dc_fast_chargers"] = np.log1p(df["dc_fast_chargers_per_1k"])
    df["chargers_per_1k"] = df["total_chargers"] * 1000 / pop
    df["y_chargers"] = np.log1p(df["chargers_per_1k"])
    df["pv_kw_per_1k"] = pv_kw_per_1000(df["PV_system_size_DC"], pop)
    df["y_pv"] = np.log1p(df["pv_kw_per_1k"])
    df["storage_mw_per_100k"] = df["storage_capacity_mw"] * 100000 / pop
    df["y_storage"] = np.log1p(df["storage_mw_per_100k"])
    df["wind_mw_per_100k"] = df["wind_capacity_mw"] * 100000 / pop
    df["y_wind_mw"] = np.log1p(df["wind_mw_per_100k"])
    df["any_turbines"] = (pd.to_numeric(df["wind_turbine_count"], errors="coerce").fillna(0) > 0).astype(float)
    df["plant_mw_per_100k"] = df["plant_capacity_mw"] * 100000 / pop
    df["wind_mw_per_100k_ctrl"] = df["wind_capacity_mw"] * 100000 / pop
    df["turbines_per_100k"] = df["wind_turbine_count"] * 100000 / pop
    df["log_median_household_income"] = np.log(df["median_household_income"].where(df["median_household_income"] > 0))
    df["log_median_housing_value"] = np.log(df["median_housing_value"].where(df["median_housing_value"] > 0))
    df["combined_nonwhite_share"] = df[["pct_black", "pct_hispanic", "pct_asian"]].sum(axis=1, min_count=1)
    df["energy_burden_pct"] = pd.to_numeric(df["energy_burden_pct"], errors="coerce")
    df["energy_gap_per_capita"] = df["energy_affordability_gap"] / pop
    df["log_energy_gap_per_capita"] = np.log1p(df["energy_gap_per_capita"])
    return df

In [ ]:
#### Data load - identical to regression.ipynb's data-prep cell.
df = pd.read_csv("../data/processed/combined_der_dataset_w_controls_predictors.csv")
df.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
df.rename(columns={
    "ghi_mean_kwh_m2_day_2024": "ghi_mean_kwh_m2_day_2023",
    "wind_ws10m_mean_2024": "wind_ws10m_mean_2023",
    "wind_ws50m_mean_2024": "wind_ws50m_mean_2023",
}, inplace=True)
for c in [
    "median_household_income", "poverty_rate", "pct_bachelors_plus",
    "pct_black", "pct_hispanic", "pct_asian", "median_housing_value",
    "pct_single_family_units", "pct_multifamily_units", "pct_mobile_home_units",
    "pct_other_housing_units", "owner_occupied_rate",
    "cdd65_2023", "hdd65_2023", "t2m_mean_c_2023",
    "ghi_mean_kwh_m2_day_2023", "wind_ws10m_mean_2023", "wind_ws50m_mean_2023",
    "total_population", "lat", "lon", "log_kwh",
    "plant_capacity_mw", "storage_capacity_mw", "wind_capacity_mw", "wind_turbine_count",
    "PV_system_size_DC", "total_chargers", "level1_chargers", "level2_chargers", "dc_fast_chargers",
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["utility_type"] = df["utility_type"].fillna("POU")
df = prep_outcomes_per_capita(df, min_pop=1000)

core_needed = ["median_household_income", "pct_black", "pct_hispanic", "pct_asian"]
required_housing = [
    "pct_single_family_units", "pct_multifamily_units",
    "pct_mobile_home_units", "pct_other_housing_units", "owner_occupied_rate",
]
missing_housing = [c for c in required_housing if c not in df.columns]
if missing_housing:
    raise ValueError(f"Release dataset is missing housing controls: {missing_housing}")
df = df.dropna(subset=[c for c in core_needed if c in df.columns]).copy()

print(f"Loaded {len(df)} ZIP/ZCTAs after core dropna.")

In [ ]:
# Control-block definitions - identical to CONFOUNDER_BLOCKS in regression.ipynb.
income = "log_median_household_income"
race = ["pct_black", "pct_hispanic", "pct_asian"]
controls_common = ["poverty_rate"]
controls_3A = ["cdd65_2023", "hdd65_2023"]
controls_3C = ["ghi_mean_kwh_m2_day_2023"]
controls_3D = ["wind_ws50m_mean_2023"]
ses_bach = ["pct_bachelors_plus"]
ses_house = ["log_median_housing_value"]
housing_structure = ["pct_multifamily_units", "pct_mobile_home_units", "pct_other_housing_units"]
tenure = ["owner_occupied_rate"]
utility_fe = "C(utility)"
demand_proxy = "log_kwh"
energy_burden_features = ["energy_burden_pct", "log_energy_gap_per_capita"]


def _fips5(value):
    if pd.isna(value):
        return pd.NA
    return str(int(float(value))).zfill(5)


county_values = df["county_geoid"].map(_fips5)
df["county_geoid"] = county_values.astype(object).where(county_values.notna(), np.nan)
county_fe = "C(county_geoid)"

CONFOUNDER_BLOCKS = {
    "education": ses_bach,
    "housing_value": ses_house,
    "housing_structure": housing_structure,
    "tenure": tenure,
    "utility_fe": [utility_fe],
    "county_fe": [county_fe],
}
SPEC_CURVE_FOCAL_TERMS = [income, "pct_black", "pct_hispanic", "pct_asian"]

analysis_df_raw = df.copy()

In [ ]:
def build_formula(outcome, climate_controls, extra_terms=None, fe_terms=None,
                  keepincome=True, demand_proxy_term=None, controls_common_terms=None):
    rhs = []
    if keepincome:
        rhs.append(income)
    rhs += race
    rhs += list(controls_common_terms or controls_common)
    if demand_proxy_term is not None:
        rhs.append(demand_proxy_term)
    rhs += list(climate_controls or [])
    if extra_terms:
        rhs += list(extra_terms)
    if fe_terms:
        rhs += list(fe_terms)
    seen = set()
    rhs = [x for x in rhs if not (x in seen or seen.add(x))]
    return f"{outcome} ~ " + " + ".join(rhs)


def controls_for_outcome(outcome):
    if outcome == "y_pv":
        return controls_3C
    if outcome == "y_storage":
        return controls_3A
    if outcome in {"y_chargers", "y_level1_chargers", "y_level2_chargers", "y_dc_fast_chargers"}:
        return controls_3A
    if outcome in {"y_wind_mw", "any_turbines"}:
        return controls_3D
    if outcome in energy_burden_features:
        return controls_3A + controls_3C
    return []


def standardize_model_frame(source_df, outcome):
    df_model = source_df.copy()
    candidate_standardized_predictors = [
        "log_median_household_income", "log_median_housing_value", "log_pop_density",
        "poverty_rate", "pct_bachelors_plus", "pct_black", "pct_hispanic", "pct_asian",
        "combined_nonwhite_share", "pct_single_family_units", "pct_multifamily_units",
        "pct_mobile_home_units", "pct_other_housing_units", "owner_occupied_rate",
        "cdd65_2023", "hdd65_2023", "t2m_mean_c_2023", "ghi_mean_kwh_m2_day_2023",
        "wind_ws10m_mean_2023", "wind_ws50m_mean_2023", "lat", "lon", "log_kwh",
        "plant_capacity_mw", "storage_capacity_mw", "wind_capacity_mw", "wind_turbine_count",
        "PV_system_size_DC", "plant_mw_per_100k", "storage_mw_per_100k",
        "wind_mw_per_100k_ctrl", "turbines_per_100k", "energy_burden_pct",
        "energy_affordability_index", "log_energy_gap_per_capita",
        "y_pv", "y_storage", "y_chargers", "y_wind_mw",
        "y_level1_chargers", "y_level2_chargers", "y_dc_fast_chargers",
    ]
    cols_to_standardize = [
        c for c in candidate_standardized_predictors
        if c in df_model.columns and c != outcome and df_model[c].nunique(dropna=True) > 1
    ]
    if cols_to_standardize:
        df_model[cols_to_standardize] = StandardScaler().fit_transform(df_model[cols_to_standardize])
    return df_model


def export_result_table(res, title, table_dir):
    table_dir.mkdir(parents=True, exist_ok=True)
    res.summary2().tables[1].to_csv(table_dir / f"{title}.csv")
    # Same companion-file convention regression.ipynb's main() already uses on main:
    # long "statistic,value" rows, so a reader parsing either notebook's fit-stats
    # files sees one format.
    stats = fit_stats(res)
    pd.DataFrame({"statistic": list(stats), "value": list(stats.values())}).to_csv(
        table_dir / f"{title} fit stats.csv", index=False
    )
    return stats


def export_vif_table(formula, df_model, title, table_dir):
    vif = vif_from_formula(formula, df_model)
    vif.to_csv(table_dir / f"{title} VIF.csv", index=False)
    return vif

In [ ]:
def run_ladder_suite(outcome, source_df, standardized_flag):
    """Fit Models C1-C5, S, O for one outcome/mode. Returns the raw-mode Oster rows.

    Formula construction is identical to regression.ipynb's cumulative-ladder block
    (verified: the two outcomes it already covers, y_pv and y_storage, produce
    byte-identical coefficient and VIF tables to this notebook's output).
    """
    table_dir = OUTPUT_TABLE_DIRS[standardized_flag]
    mode_label = "standardized" if standardized_flag else "raw"
    controls_cur = controls_for_outcome(outcome)
    df_model = standardize_model_frame(source_df, outcome) if standardized_flag else source_df.copy()

    charger_exclusions = [
        "total_chargers", "level1_chargers", "level2_chargers", "dc_fast_chargers",
        "chargers_per_1k", "level1_chargers_per_1k", "level2_chargers_per_1k",
        "dc_fast_chargers_per_1k", "y_chargers", "y_level1_chargers",
        "y_level2_chargers", "y_dc_fast_chargers",
    ]
    exclude_for_y = {
        "y_chargers": charger_exclusions,
        "y_level1_chargers": charger_exclusions,
        "y_level2_chargers": charger_exclusions,
        "y_dc_fast_chargers": charger_exclusions,
        "y_pv": ["PV_system_size_DC", "pv_kw_per_1k", "y_pv"],
        "y_storage": ["storage_capacity_mw", "storage_mw_per_100k"],
        "y_wind_mw": ["wind_capacity_mw", "wind_mw_per_100k"],
        "any_turbines": ["wind_capacity_mw", "wind_mw_per_100k", "wind_turbine_count",
                          "wind_mw_per_100k_ctrl", "turbines_per_100k", "any_turbines"],
    }
    infra_candidates = ["plant_capacity_mw", "storage_capacity_mw", "wind_capacity_mw",
                         "wind_turbine_count", "PV_system_size_DC"]
    infra = [c for c in infra_candidates if c in df_model.columns and c not in exclude_for_y.get(outcome, [])]

    c2_terms = CONFOUNDER_BLOCKS["education"] + CONFOUNDER_BLOCKS["housing_value"]
    c3_terms = c2_terms + CONFOUNDER_BLOCKS["housing_structure"] + CONFOUNDER_BLOCKS["tenure"]

    fc1 = build_formula(outcome, climate_controls=controls_cur, controls_common_terms=["poverty_rate"])
    fc2 = build_formula(outcome, climate_controls=controls_cur, extra_terms=c2_terms, controls_common_terms=["poverty_rate"])
    fc3 = build_formula(outcome, climate_controls=controls_cur, extra_terms=c3_terms, controls_common_terms=["poverty_rate"])
    fc4 = build_formula(
        outcome, climate_controls=controls_cur, extra_terms=c3_terms,
        fe_terms=CONFOUNDER_BLOCKS["utility_fe"], controls_common_terms=["poverty_rate"],
    )
    fc5 = build_formula(
        outcome, climate_controls=[], extra_terms=c3_terms,
        fe_terms=CONFOUNDER_BLOCKS["utility_fe"] + CONFOUNDER_BLOCKS["county_fe"],
        controls_common_terms=["poverty_rate"],
    )
    fo = build_formula(
        outcome, climate_controls=[],
        extra_terms=c3_terms + [demand_proxy] + infra,
        fe_terms=CONFOUNDER_BLOCKS["utility_fe"] + CONFOUNDER_BLOCKS["county_fe"],
        controls_common_terms=["poverty_rate"],
    )

    ladder_index = common_sample_index([fc1, fc2, fc3, fc4, fc5, fo], df_model, cluster_col="county_geoid")
    if len(ladder_index) < 100:
        print(f"Skipping {outcome} ({mode_label}): frozen sample is only {len(ladder_index)} rows.")
        return []

    df_ladder = df_model.loc[ladder_index]

    ladder_specs = [
        ("Model C1 (core, common sample)", fc1, None),
        ("Model C2 (+ education, housing value)", fc2, None),
        ("Model C3 (+ housing structure, tenure)", fc3, None),
        ("Model C4 (+ utility FE)", fc4, None),
        ("Model C5 (+ county FE, county-clustered SEs)", fc5, "county_geoid"),
        ("Model S (saturated confounders)", fc5, "county_geoid"),
        ("Model O (over-controlled: + demand and infrastructure)", fo, "county_geoid"),
    ]
    results = {}
    nobs_seen = set()
    for label, formula, cluster in ladder_specs:
        res = run_ols(formula, df_ladder, cluster_col=cluster)
        export_result_table(res, f"{outcome} | {label}", table_dir)
        export_vif_table(formula, df_ladder, f"{outcome} | {label}", table_dir)
        results[label] = res
        nobs_seen.add(int(res.nobs))

    if len(nobs_seen) != 1:
        raise ValueError(f"Ladder for {outcome} ({mode_label}) is not on a frozen sample: {nobs_seen}")

    print(f"{outcome} ({mode_label}): ladder frozen sample = {df_ladder.shape[0]} ZIPs, nobs = {nobs_seen.pop()}")

    oster_rows = []
    if not standardized_flag:
        res_c1 = results["Model C1 (core, common sample)"]
        res_s = results["Model S (saturated confounders)"]
        for term in SPEC_CURVE_FOCAL_TERMS:
            if term not in res_c1.params.index or term not in res_s.params.index:
                continue
            delta = oster_delta(res_c1, res_s, term)
            oster_rows.append({
                "outcome": outcome,
                "term": term,
                "beta_restricted": res_c1.params[term],
                "beta_full": res_s.params[term],
                "r2_restricted": res_c1.rsquared,
                "r2_full": res_s.rsquared,
                "rmax": min(1.3 * res_s.rsquared, 1.0),
                "delta": delta,
            })
    return oster_rows

In [ ]:
all_oster_rows = []
for outcome in OUTCOMES_TO_RUN:
    all_oster_rows.extend(run_ladder_suite(outcome, analysis_df_raw, False))
    if RUN_BOTH_OUTPUT_MODES:
        run_ladder_suite(outcome, analysis_df_raw, True)

oster_df = pd.DataFrame(all_oster_rows)
OSTER_DELTA_PATH.parent.mkdir(parents=True, exist_ok=True)
oster_df.to_csv(OSTER_DELTA_PATH, index=False)
print(f"Saved Oster delta bounds ({len(oster_df)} rows) to {OSTER_DELTA_PATH}")
oster_df

## Specification curve

Every subset of the six confounder blocks, raw scale only. Combinations that stack
both `county_fe` and `utility_fe` are skipped (Sec. 2 of the plan: both are
near-exhaustive geographic/institutional partitions of the same ZIPs, and stacking
them is close to collinear), leaving `2**6 - 2**4 = 48` valid specifications per
outcome, not the full 64.

In [ ]:
spec_curve_rows = []
if RUN_SPEC_CURVE:
    block_names = list(CONFOUNDER_BLOCKS.keys())

    for outcome in OUTCOMES_TO_RUN:
        controls_cur = controls_for_outcome(outcome)

        for r in range(len(block_names) + 1):
            for combo in itertools.combinations(block_names, r):
                combo_set = set(combo)
                if {"county_fe", "utility_fe"} <= combo_set:
                    continue
                climate_cur = [] if "county_fe" in combo_set else controls_cur
                extra_terms, fe_terms = [], []
                for name in combo:
                    if name in {"utility_fe", "county_fe"}:
                        fe_terms += CONFOUNDER_BLOCKS[name]
                    else:
                        extra_terms += CONFOUNDER_BLOCKS[name]

                formula = build_formula(
                    outcome, climate_controls=climate_cur, extra_terms=extra_terms,
                    fe_terms=fe_terms, controls_common_terms=["poverty_rate"],
                )
                res = run_ols(formula, analysis_df_raw)

                for term in SPEC_CURVE_FOCAL_TERMS:
                    if term not in res.params.index:
                        continue
                    spec_curve_rows.append({
                        "outcome": outcome,
                        "blocks": "+".join(sorted(combo)) if combo else "none",
                        "term": term,
                        "coef": res.params[term],
                        "se": res.bse[term],
                        "pval": res.pvalues[term],
                        "nobs": int(res.nobs),
                        "rsquared": res.rsquared,
                    })

    spec_curve_df = pd.DataFrame(spec_curve_rows)
    SPEC_CURVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    spec_curve_df.to_csv(SPEC_CURVE_PATH, index=False)
    print(f"Saved spec curve ({len(spec_curve_df)} rows) to {SPEC_CURVE_PATH}")
    display(spec_curve_df.groupby(["outcome", "term"]).size().rename("n_specs"))
else:
    print("RUN_SPEC_CURVE = False: skipped.")